In [1]:
"""
from google.colab import drive
drive.mount('/content/drive')"""

"\nfrom google.colab import drive\ndrive.mount('/content/drive')"

In [2]:
"""import kagglehub
import os

if os.path.exists('/content/drive'):
    output_dir = '/content/drive/MyDrive/sample data/stanford_dogs_dataset'
else:
    output_dir = "D:\Code With Harry\Intro-To-Deeplearing\PYTORCH\sample data\stanford_dogs_dataset"


path = kagglehub.dataset_download("jessicali9530/stanford-dogs-dataset", output_dir=output_dir)

print("Path to dataset files:", path)"""

<>:1: SyntaxWarning: invalid escape sequence '\C'
<>:1: SyntaxWarning: invalid escape sequence '\C'
C:\Users\Dilip Das\AppData\Local\Temp\ipykernel_11972\1949221100.py:1: SyntaxWarning: invalid escape sequence '\C'
  """import kagglehub


'import kagglehub\nimport os\n\nif os.path.exists(\'/content/drive\'):\n    output_dir = \'/content/drive/MyDrive/sample data/stanford_dogs_dataset\'\nelse:\n    output_dir = "D:\\Code With Harry\\Intro-To-Deeplearing\\PYTORCH\\sample data\\stanford_dogs_dataset"\n\n\npath = kagglehub.dataset_download("jessicali9530/stanford-dogs-dataset", output_dir=output_dir)\n\nprint("Path to dataset files:", path)'

In [32]:
import os
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from PIL import Image
from torchvision import transforms
from sklearn.model_selection import train_test_split

In [33]:
torch.manual_seed(42)

In [34]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [35]:
def create_dataset(data_dir):
    images = []
    labels = []

    class_names = sorted(os.listdir(data_dir))
    for label, class_name in enumerate(class_names):
        class_dir = os.path.join(data_dir, class_name)
        
        for image_names in os.listdir(class_dir):
            image_paths = os.path.join(class_dir, image_names)

            images.append(image_paths)
            labels.append(label)
    return images, labels



In [36]:
if os.path.exists("/content/drive"):
    out_dir = "/content/drive/MyDrive/sample data/stanford_dogs_dataset/images/Images"
else:
    out_dir = "../../sample data/stanford_dogs_dataset/images/Images"

X, y = create_dataset(out_dir)


X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42, stratify=y)

In [37]:
class Dogs(Dataset):

    def __init__(self, images, labels, transforms = None):

        self.images = images
        self.labels = labels

        self.transforms = transforms
    
    def __len__(self):
        lenght = len(self.images)
        return lenght
    
    def __getitem__(self, index):
        image_path = self.images[index]
        image = Image.open(image_path).convert("RGB")

        label = self.labels[index]

        if self.transforms :
            image = self.transforms(image)

        return image, label



In [38]:
train_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((227, 227)),
    transforms.RandomHorizontalFlip(p = 0.5),
    transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])





def alex_test_ten_crop(crops):

    cropped = []

    for crop in crops:
        tensor_crop = transforms.ToTensor()(crop)
        
        cropped.append(tensor_crop)
    return torch.stack(cropped)


test_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.TenCrop((227, 227)),
    transforms.Lambda(alex_test_ten_crop),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [39]:
# daata loader

train_dataset = Dogs(X_train, y_train, transforms=train_transforms)
test_dataset = Dogs(X_test, y_test, transforms= test_transforms)

In [40]:
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=True)

In [41]:
#help(nn.Conv2d)

In [42]:
print(len(set(y)))
total_classes = len(set(y))

120


In [43]:
class Alexnet(nn.Module):

    def __init__(self, input_features, total_classes):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(in_channels=input_features, out_channels=96, kernel_size=11, stride=4),
            nn.BatchNorm2d(96),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3,stride=2),

            nn.Conv2d(in_channels=96, out_channels=256, kernel_size=5, stride=1, padding=2),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2),

            nn.Conv2d(in_channels=256, out_channels=384, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=384, out_channels=384, kernel_size=3, padding=1),
            nn.ReLU(),

            nn.Conv2d(in_channels=384, out_channels=256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(in_features=256*6*6, out_features=4096),
            nn.ReLU(),

            nn.Dropout(p=0.5),
            nn.Linear(in_features=4096, out_features=4096),
            nn.ReLU(),

            nn.Linear(in_features=4096, out_features= total_classes)
        )
    
    def forward(self,x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x



In [44]:
# CALCULATING the Input for Fully Connected layer/ self.Classifier

images, labels = next(iter(train_dataloader))
print(images.shape)
model = Alexnet(input_features=images.shape[1], total_classes=len(set(y)))


print()

x = torch.randn(32, 3, 227, 227)
x = model.features(x)
print(x.shape)
print(x.flatten(1).shape)

torch.Size([32, 3, 227, 227])

torch.Size([32, 256, 6, 6])
torch.Size([32, 9216])


In [45]:
model.to(device)

Alexnet(
  (features): Sequential(
    (0): Conv2d(3, 96, kernel_size=(11, 11), stride=(4, 4))
    (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(96, 256, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (5): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(256, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): ReLU()
    (10): Conv2d(384, 384, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU()
    (12): Conv2d(384, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU()
    (14): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Li

In [46]:
from torchinfo import summary
summary(model, input_size=(32, 3, 227, 227))

Layer (type:depth-idx)                   Output Shape              Param #
Alexnet                                  [32, 120]                 --
├─Sequential: 1-1                        [32, 256, 6, 6]           --
│    └─Conv2d: 2-1                       [32, 96, 55, 55]          34,944
│    └─BatchNorm2d: 2-2                  [32, 96, 55, 55]          192
│    └─ReLU: 2-3                         [32, 96, 55, 55]          --
│    └─MaxPool2d: 2-4                    [32, 96, 27, 27]          --
│    └─Conv2d: 2-5                       [32, 256, 27, 27]         614,656
│    └─BatchNorm2d: 2-6                  [32, 256, 27, 27]         512
│    └─ReLU: 2-7                         [32, 256, 27, 27]         --
│    └─MaxPool2d: 2-8                    [32, 256, 13, 13]         --
│    └─Conv2d: 2-9                       [32, 384, 13, 13]         885,120
│    └─ReLU: 2-10                        [32, 384, 13, 13]         --
│    └─Conv2d: 2-11                      [32, 384, 13, 13]         1,

In [47]:
# CAlculation Output shape
dummy = torch.zeros(32, 3, 227,227).to(device)
output = model.features(dummy)
print(output.shape)

print(output.flatten(1).shape)

torch.Size([32, 256, 6, 6])
torch.Size([32, 9216])


In [ ]:
import torch.optim as optim
learning_rate = 0.01
epochs = 25
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(
    model.parameters(), 
    lr=learning_rate,          # Learning rate (originally decreased by 10 when validation error plateaued)
    momentum=0.9,     # Classic momentum parameter
    weight_decay=0.0005 # L2 regularization to prevent overfitting
)

In [20]:
for epoch in range(epochs):
    total_epochs_loss = 0
    for batch_features, batch_labels in train_dataloader:

            
        batch_features, batch_labels = batch_features.to(device), batch_labels.long().to(device)

        output = model(batch_features)

        loss = criterion(output, batch_labels)

        optimizer.zero_grad()
        loss.backward()

        optimizer.step()

        total_epochs_loss = total_epochs_loss + loss.item()
    avg_loss = total_epochs_loss/len(train_dataloader)
    print(f'Epoch: {epoch + 1} , Loss: {avg_loss}')

Epoch: 1 , Loss: 4.773134345452762
Epoch: 2 , Loss: 4.58342614590543
Epoch: 3 , Loss: 4.393525445808485
Epoch: 4 , Loss: 4.264632916219026
Epoch: 5 , Loss: 4.163560235847547
Epoch: 6 , Loss: 4.067028353978129
Epoch: 7 , Loss: 3.964832426737813
Epoch: 8 , Loss: 3.8689612624714678
Epoch: 9 , Loss: 3.7603226434837267
Epoch: 10 , Loss: 3.664896210420479
Epoch: 11 , Loss: 3.563338399627834
Epoch: 12 , Loss: 3.4561043711541926
Epoch: 13 , Loss: 3.3680145907170562
Epoch: 14 , Loss: 3.268824232897712
Epoch: 15 , Loss: 3.1828248135094506
Epoch: 16 , Loss: 3.072676141516676
Epoch: 17 , Loss: 2.9934324236749443
Epoch: 18 , Loss: 2.89029961257305
Epoch: 19 , Loss: 2.8214720932025354
Epoch: 20 , Loss: 2.7343171184502757
Epoch: 21 , Loss: 2.6519995890774775
Epoch: 22 , Loss: 2.5753070757227037
Epoch: 23 , Loss: 2.5116975092193456
Epoch: 24 , Loss: 2.439583382097263
Epoch: 25 , Loss: 2.3875637841456143


In [21]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for batch_features, batch_labels in train_dataloader:

        batch_features = batch_features.to(device)
        batch_labels = batch_labels.long().to(device)

        output = model(batch_features)

        predicted = output.argmax(dim=1)

        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

accuracy = 100 * correct / total

print(f"Training Accuracy: {accuracy:.2f}%")

Training Accuracy: 42.47%


In [31]:
correct = 0
total = 0

model.eval()

with torch.no_grad():

    for batch_features, batch_labels in test_dataloader:
        #print(batch_features.shape)
        batch_size, num_crops, C, H, W = batch_features.shape

        batch_features = batch_features.view(batch_size * num_crops,C,H,W).to(device)

        #print(batch_features.shape)

        batch_labels = batch_labels.long().to(device)

        output = model(batch_features)
        output = output.reshape(batch_size, num_crops, -1)
        #print(output.shape)
        
        output = output.mean(dim=1)
        predicted = output.argmax(dim=1)
        
        #print(predicted.shape)
        
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum()

accuracy = 100 * correct / total

print(f"Testing Accuracy: {accuracy:.2f}%")

Testing Accuracy: 33.48%


| Optimizer          |   LR | Epochs |   Loss | Train Acc | Test Acc | Regularization |
| ------------------ | ---: | -----: | -----: | --------: | -------: | -------------- |
| **SGD + Momentum** | 0.01 |     25 | 2.3876 |    42.47% |   33.48% | L2 = 0.0005    |
